# 🎬 CineMatch — AI Movie Recommender
### Run & Deploy Streamlit App from Google Colab

This notebook will:
1. Install all dependencies
2. Upload your dataset (`cleaned_imdb.csv`)
3. Create the Streamlit app file
4. Launch the app with a **public URL** using `localtunnel`

---
> **Dataset required:** `cleaned_imdb.csv` (838 IMDB movies)

## ✅ Step 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q streamlit pandas numpy scikit-learn plotly matplotlib seaborn
!pip install -q pyngrok  # for public URL tunneling (alternative method)
!npm install -q -g localtunnel
print('✅ All packages installed!')

## ✅ Step 2 — Upload Your Dataset

In [ ]:
from google.colab import files
import os

print('📁 Please upload cleaned_imdb.csv')
uploaded = files.upload()

if 'cleaned_imdb.csv' in uploaded:
    print('✅ Dataset uploaded successfully!')
    import pandas as pd
    df = pd.read_csv('cleaned_imdb.csv')
    print(f'   Shape: {df.shape}')
    print(f'   Columns: {df.columns.tolist()}')
else:
    print('⚠️ Please make sure to upload cleaned_imdb.csv')

## ✅ Step 3 — Create the Streamlit App File

This cell writes the full app code to `app.py`

In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="CineMatch – AI Movie Recommender",
    page_icon="🎬",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
@import url("https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700&family=DM+Sans:wght@300;400;500&display=swap");
html, body, [class*="css"] { font-family: "DM Sans", sans-serif; }
.stApp { background: linear-gradient(135deg, #0d0d14 0%, #12121f 100%); }
h1,h2,h3 { font-family: "Playfair Display", serif !important; }
.movie-card { background:rgba(255,255,255,0.04); border:1px solid rgba(255,255,255,0.08); border-radius:14px; padding:1.4rem; margin-bottom:1rem; }
.movie-title { font-family:"Playfair Display",serif; font-size:1.2rem; color:#e8c97e; margin-bottom:0.3rem; }
.movie-meta { font-size:0.82rem; color:#8888aa; margin-bottom:0.5rem; }
.badge { display:inline-block; background:rgba(232,201,126,0.15); color:#e8c97e; border-radius:20px; padding:2px 10px; font-size:0.75rem; margin-right:4px; }
.stButton > button { background:linear-gradient(135deg,#e8c97e,#c46b3a) !important; color:#0d0d14 !important; font-weight:600 !important; border:none !important; border-radius:8px !important; width:100% !important; }
label { color:#c8c8dd !important; }
</style>
""", unsafe_allow_html=True)

@st.cache_data
def load_data():
    df = pd.read_csv("cleaned_imdb.csv")
    df = df.dropna(subset=["Title","Genre","Rating","Year","Director"])
    df["Genre"] = df["Genre"].fillna("")
    df["Director"] = df["Director"].fillna("")
    df["Actors"] = df["Actors"].fillna("")
    df["Description"] = df["Description"].fillna("")
    df["Metascore"] = df["Metascore"].fillna(df["Metascore"].median())
    return df

@st.cache_resource
def build_model(df):
    df = df.copy()
    df["text_soup"] = (
        df["Genre"].str.replace(","," ") + " " +
        df["Director"].str.replace(" ","") + " " +
        df["Actors"].str.replace(",","").str.replace(" ","") + " " +
        df["Description"]
    )
    tfidf = TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1,2))
    tfidf_matrix = tfidf.fit_transform(df["text_soup"])
    scaler = MinMaxScaler()
    num_feats = scaler.fit_transform(df[["Rating","Year","Votes","Runtime_(Minutes)"]].fillna(0))
    return tfidf_matrix, num_feats, df.reset_index(drop=True)

def get_recommendations(title, df, tfidf_matrix, num_feats,
                        genre_filter=None, year_range=None,
                        rating_min=0.0, director_filter=None, top_n=10):
    matches = df[df["Title"].str.lower() == title.lower()]
    if matches.empty:
        return None, "Movie not found."
    idx = matches.index[0]
    text_sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    num_sim = cosine_similarity(num_feats[idx].reshape(1,-1), num_feats).flatten()
    scores = 0.70 * text_sim + 0.30 * num_sim
    scores[idx] = -1
    sim_df = df.copy()
    sim_df["similarity_score"] = scores
    if genre_filter:
        sim_df = sim_df[sim_df["Genre"].apply(lambda g: any(gf.lower() in g.lower() for gf in genre_filter))]
    if year_range:
        sim_df = sim_df[(sim_df["Year"] >= year_range[0]) & (sim_df["Year"] <= year_range[1])]
    if rating_min:
        sim_df = sim_df[sim_df["Rating"] >= rating_min]
    if director_filter and director_filter != "Any":
        sim_df = sim_df[sim_df["Director"] == director_filter]
    result = sim_df.sort_values("similarity_score", ascending=False).head(top_n)
    return result, None

def smart_search(genre_list, year_range, rating_min, director, df, top_n=10):
    filtered = df.copy()
    if genre_list:
        filtered = filtered[filtered["Genre"].apply(lambda g: any(gf.lower() in g.lower() for gf in genre_list))]
    filtered = filtered[(filtered["Year"] >= year_range[0]) & (filtered["Year"] <= year_range[1]) & (filtered["Rating"] >= rating_min)]
    if director and director != "Any":
        filtered = filtered[filtered["Director"] == director]
    if filtered.empty:
        return filtered
    filtered = filtered.copy()
    filtered["rank_score"] = 0.6 * filtered["Rating"] / 10 + 0.4 * (np.log1p(filtered["Votes"]) / np.log1p(filtered["Votes"].max()))
    return filtered.sort_values("rank_score", ascending=False).head(top_n)

df = load_data()
tfidf_matrix, num_feats, df_indexed = build_model(df)

ALL_GENRES = sorted(["Action","Adventure","Animation","Biography","Comedy","Crime","Drama","Fantasy","Horror","Mystery","Sci-Fi","Thriller"])
ALL_DIRECTORS = ["Any"] + sorted(df["Director"].unique().tolist())
ALL_TITLES = sorted(df["Title"].unique().tolist())

st.markdown("<h1 style=\'color:#e8c97e;font-family:Playfair Display,serif\'>🎬 CineMatch</h1>", unsafe_allow_html=True)
st.markdown("<p style=\'color:#8888aa\'>AI-powered movie recommendations · IMDB dataset · ML similarity engine</p>", unsafe_allow_html=True)

c1,c2,c3,c4 = st.columns(4)
c1.metric("Movies", len(df))
c2.metric("Directors", df["Director"].nunique())
c3.metric("Genres", len(ALL_GENRES))
c4.metric("Years", f"{df["Year"].min()}–{df["Year"].max()}")

st.markdown("---")

with st.sidebar:
    st.markdown("## 🎛️ Filters")
    mode = st.radio("Mode", ["🎯 Find Similar Movies", "🔍 Smart Search"])
    genre_filter = st.multiselect("Genre", ALL_GENRES, default=[])
    year_range = st.slider("Year Range", int(df["Year"].min()), int(df["Year"].max()), (2008, 2016))
    rating_min = st.slider("Min Rating", 1.0, 9.0, 6.0, 0.1)
    director_filter = st.selectbox("Director", ALL_DIRECTORS)
    top_n = st.slider("Results", 3, 20, 8)

tab1, tab2, tab3 = st.tabs(["🎯 Recommendations", "📊 Data Explorer", "🧠 Model Insights"])

with tab1:
    if "🎯 Find Similar" in mode:
        st.subheader("Find Similar Movies")
        selected_movie = st.selectbox("Choose a movie you like:", ALL_TITLES)
        if st.button("Get Recommendations 🚀"):
            seed = df[df["Title"] == selected_movie].iloc[0]
            with st.expander(f"ℹ️ {selected_movie}", expanded=True):
                st.write(f"**Director:** {seed["Director"]} | **Year:** {int(seed["Year"])} | **Rating:** ⭐ {seed["Rating"]}")
                st.write(f"**Genre:** {seed["Genre"]}")
                st.write(f"**Cast:** {seed["Actors"]}")
                st.write(seed["Description"])
            recs, err = get_recommendations(selected_movie, df_indexed, tfidf_matrix, num_feats,
                genre_filter if genre_filter else None, year_range, rating_min,
                director_filter if director_filter != "Any" else None, top_n)
            if err:
                st.error(err)
            elif recs is not None and not recs.empty:
                st.success(f"Top {len(recs)} recommendations for you!")
                for i, (_, row) in enumerate(recs.iterrows(), 1):
                    score_pct = int(row["similarity_score"] * 100)
                    with st.container():
                        st.markdown(f"**#{i} {row["Title"]}** ({int(row["Year"])}) — ⭐ {row["Rating"]} | 🎬 {row["Director"]} | 🎭 {row["Genre"]}")
                        st.progress(score_pct, text=f"Match: {score_pct}%")
                        st.caption(str(row["Description"])[:180])
                        st.divider()
            else:
                st.warning("No movies found. Try relaxing your filters.")
    else:
        st.subheader("Smart Search by Preferences")
        if st.button("🔍 Search Movies"):
            results = smart_search(genre_filter if genre_filter else None, year_range, rating_min, director_filter, df_indexed, top_n)
            if results.empty:
                st.warning("No movies match your filters.")
            else:
                st.success(f"Found {len(results)} movies!")
                for i, (_, row) in enumerate(results.iterrows(), 1):
                    st.markdown(f"**#{i} {row["Title"]}** ({int(row["Year"])}) — ⭐ {row["Rating"]} | 🎬 {row["Director"]}")
                    st.caption(str(row["Description"])[:180])
                    st.divider()

with tab2:
    st.subheader("Dataset Explorer")
    col1, col2 = st.columns(2)
    with col1:
        fig1 = px.histogram(df, x="Rating", nbins=30, title="Rating Distribution", color_discrete_sequence=["#e8c97e"])
        fig1.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#c8c8dd")
        st.plotly_chart(fig1, use_container_width=True)
    with col2:
        genre_counts = df["Main_Genre"].value_counts().reset_index()
        genre_counts.columns = ["Genre","Count"]
        fig2 = px.bar(genre_counts, x="Count", y="Genre", orientation="h", title="Movies per Genre", color="Count", color_continuous_scale=["#c46b3a","#e8c97e"])
        fig2.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#c8c8dd", coloraxis_showscale=False)
        st.plotly_chart(fig2, use_container_width=True)
    col3, col4 = st.columns(2)
    with col3:
        fig3 = px.box(df, x="Main_Genre", y="Rating", color="Main_Genre", title="Rating by Genre")
        fig3.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#c8c8dd", showlegend=False)
        st.plotly_chart(fig3, use_container_width=True)
    with col4:
        year_counts = df.groupby("Year").size().reset_index(name="Count")
        fig4 = px.line(year_counts, x="Year", y="Count", title="Movies Per Year", markers=True, color_discrete_sequence=["#e8c97e"])
        fig4.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#c8c8dd")
        st.plotly_chart(fig4, use_container_width=True)
    fig5 = px.scatter(df, x="Rating", y="Revenue(Crores)", color="Main_Genre", hover_name="Title", size="Votes", title="Rating vs Revenue")
    fig5.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#c8c8dd")
    st.plotly_chart(fig5, use_container_width=True)
    st.subheader("Raw Dataset")
    search = st.text_input("Search title")
    show = df if not search else df[df["Title"].str.contains(search, case=False, na=False)]
    st.dataframe(show[["Title","Genre","Director","Year","Rating","Runtime_(Minutes)","Votes","Revenue(Crores)"]], use_container_width=True)

with tab3:
    st.subheader("How the ML Model Works")
    st.markdown("""
    ### Algorithm: Content-Based Filtering
    | Component | Weight | Description |
    |-----------|--------|-------------|
    | **TF-IDF Text Similarity** | 70% | Genre + Director + Actors + Description vectorized |
    | **Numerical Feature Similarity** | 30% | Rating, Year, Votes, Runtime normalized |

    **Formula:** `score = 0.70 × cosine_sim(text) + 0.30 × cosine_sim(numerics)`
    """)
    dir_stats = df.groupby("Director").agg(Avg_Rating=("Rating","mean"), Movies=("Title","count")).reset_index()
    dir_stats = dir_stats[dir_stats["Movies"] >= 2].sort_values("Avg_Rating", ascending=False).head(15)
    fig6 = px.bar(dir_stats, x="Avg_Rating", y="Director", orientation="h", title="Top 15 Directors by Avg Rating", color="Avg_Rating", color_continuous_scale=["#c46b3a","#e8c97e"])
    fig6.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#c8c8dd", coloraxis_showscale=False, yaxis={"categoryorder":"total ascending"})
    st.plotly_chart(fig6, use_container_width=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('✅ app.py created!')
print('   Lines:', len(app_code.splitlines()))

## ✅ Step 4 — Quick ML Model Test (without Streamlit)

Run this to verify the recommendation engine works before launching the app.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# Load data
df = pd.read_csv('cleaned_imdb.csv')
df = df.dropna(subset=['Title', 'Genre', 'Rating', 'Year', 'Director'])
df['text_soup'] = (
    df['Genre'].str.replace(',', ' ') + ' ' +
    df['Director'].str.replace(' ', '') + ' ' +
    df['Actors'].fillna('').str.replace(',', '').str.replace(' ', '') + ' ' +
    df['Description'].fillna('')
)

# Build TF-IDF model
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['text_soup'])

# Numerical features
scaler = MinMaxScaler()
num_feats = scaler.fit_transform(df[['Rating', 'Year', 'Votes', 'Runtime_(Minutes)']].fillna(0))
df = df.reset_index(drop=True)

print('✅ Model built!')
print(f'   TF-IDF Matrix shape: {tfidf_matrix.shape}')
print(f'   Vocabulary size: {len(tfidf.vocabulary_)}')

# Test recommendation
def recommend(title, top_n=5):
    matches = df[df['Title'].str.lower() == title.lower()]
    if matches.empty:
        print(f'Movie "{title}" not found!')
        return
    idx = matches.index[0]
    text_sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    num_sim = cosine_similarity(num_feats[idx].reshape(1, -1), num_feats).flatten()
    scores = 0.70 * text_sim + 0.30 * num_sim
    scores[idx] = -1
    top_indices = scores.argsort()[::-1][:top_n]
    print(f'\n🎬 Movies similar to "{title}":')
    print('-' * 60)
    for rank, i in enumerate(top_indices, 1):
        row = df.iloc[i]
        print(f"#{rank}: {row['Title']} ({int(row['Year'])}) | ⭐{row['Rating']} | {row['Genre']}")
        print(f"     Director: {row['Director']} | Match: {scores[i]*100:.1f}%")

# Test with sample movies
recommend('The Dark Knight')
print()
recommend('Inception')

## ✅ Step 5 — Launch the Streamlit App

This will give you a **public URL** to access your app from anywhere!

In [ ]:
import subprocess
import threading
import time

# Get the public IP of this Colab instance
!curl -s ipv4.icanhazip.com

In [ ]:
# 🚀 Launch Streamlit in background
!nohup streamlit run app.py --server.port 8501 --server.headless true &
time.sleep(5)
print('✅ Streamlit is starting...')

In [ ]:
# 🌐 Create public URL with localtunnel
# Copy the IP from the cell above and paste it when prompted by localtunnel
import subprocess

result = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

time.sleep(4)
output = result.stdout.readline()
print('🎉 Your app is live at:')
print(output.strip())
print()
print('⚠️ Note: When you open the URL, enter the Colab IP shown above as the password.')

## 🔄 Alternative: Deploy to Streamlit Cloud (Permanent Hosting)

For a permanent, always-on deployment:

1. **Create a GitHub repo** with these files:
   - `app.py`
   - `requirements.txt`
   - `cleaned_imdb.csv`

2. **Go to** [share.streamlit.io](https://share.streamlit.io)

3. **Connect your GitHub repo** and click **Deploy**

Your app will have a permanent URL like: `https://yourapp.streamlit.app`

In [ ]:
# 📦 Optional: Download all project files as ZIP for GitHub upload
import zipfile
import os
from google.colab import files

with zipfile.ZipFile('cinematch_project.zip', 'w') as zf:
    zf.write('app.py')
    zf.write('cleaned_imdb.csv')
    # Write requirements.txt
    with open('requirements.txt', 'w') as f:
        f.write("""streamlit==1.35.0
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.5.0
matplotlib==3.9.0
seaborn==0.13.2
plotly==5.22.0
""")
    zf.write('requirements.txt')

print('✅ cinematch_project.zip created!')
files.download('cinematch_project.zip')
print('📁 ZIP downloaded — upload to GitHub to deploy on Streamlit Cloud!')

## 📊 Bonus: EDA & Model Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('cleaned_imdb.csv')
sns.set_theme(style='darkgrid', palette='muted')
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('CineMatch — IMDB Dataset EDA', fontsize=16, fontweight='bold')

# 1. Rating distribution
axes[0,0].hist(df['Rating'], bins=30, color='#e8c97e', edgecolor='white')
axes[0,0].set_title('Rating Distribution')
axes[0,0].set_xlabel('Rating')

# 2. Genre count
genre_counts = df['Main_Genre'].value_counts()
axes[0,1].barh(genre_counts.index, genre_counts.values, color='#c46b3a')
axes[0,1].set_title('Movies per Genre')

# 3. Rating by genre boxplot
genres = df['Main_Genre'].unique()
data_by_genre = [df[df['Main_Genre'] == g]['Rating'].values for g in genres]
axes[0,2].boxplot(data_by_genre, labels=genres)
axes[0,2].set_title('Rating by Genre')
axes[0,2].tick_params(axis='x', rotation=45)

# 4. Year trend
year_counts = df.groupby('Year').size()
axes[1,0].plot(year_counts.index, year_counts.values, marker='o', color='#e8c97e')
axes[1,0].set_title('Movies Per Year')

# 5. Rating vs Revenue
axes[1,1].scatter(df['Rating'], df['Revenue(Crores)'], alpha=0.5, color='#52c97e')
axes[1,1].set_xlabel('Rating')
axes[1,1].set_ylabel('Revenue (Crores)')
axes[1,1].set_title('Rating vs Revenue')

# 6. Top directors
dir_avg = df.groupby('Director')['Rating'].agg(['mean','count'])
top_dirs = dir_avg[dir_avg['count'] >= 2].sort_values('mean', ascending=False).head(10)
axes[1,2].barh(top_dirs.index, top_dirs['mean'], color='#e8c97e')
axes[1,2].set_title('Top Directors (avg rating, ≥2 films)')
axes[1,2].set_xlabel('Avg Rating')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved!')

In [ ]:
# Correlation heatmap
import seaborn as sns
import matplotlib.pyplot as plt

corr_cols = ['Rating', 'Year', 'Runtime_(Minutes)', 'Votes', 'Metascore', 'Revenue(Crores)']
corr = df[corr_cols].dropna().corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlOrBr', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()